### prepare object property pcs and brain responses for voxelwise encoding model

In [1]:
import os
import pandas as pd
import scipy.io as scio
from os.path import join as pjoin

basedir = r'D:\visual_project\THINGS_project'
outdir = pjoin(basedir, r'animacy_analysis\derivatives\obj_pro_analysis_I')

def load_object_property(
        obj_tsv=pjoin(basedir, r'THINGS_stimuli\THINGSplus\Metadata\Concept-specific',
                      f'objectProperties_meanRatings.tsv'),
        size_tsv=pjoin(basedir, r'THINGS_stimuli\THINGSplus\Metadata\Concept-specific',
                       f'size_meanRatings.tsv')
):
    obj_df = pd.read_csv(obj_tsv, sep='\t')[
        ['uniqueID', 'manmade_mean', 'precious_mean', 'lives_mean', 'heavy_mean', 'natural_mean', 'moves_mean',
         'grasp_mean', 'hold_mean', 'be.moved_mean', 'pleasant_mean', 'deprecatedArousing_mean']]
    # 11 object properties
    obj_df = obj_df.rename(
        columns={'manmade_mean': 'manmade', 'precious_mean': 'precious', 'lives_mean': 'animacy', 'heavy_mean': 'heavy',
                 'natural_mean': 'natural', 'moves_mean': 'moves', 'grasp_mean': 'grasp', 'hold_mean': 'hold',
                 'be.moved_mean': 'be_moved', 'pleasant_mean': 'pleasant',
                 'deprecatedArousing_mean': 'deprecatedArousing'})
    obj_df.head()
    # size dimension
    size_df = pd.read_csv(size_tsv, sep='\t')[['uniqueID', 'Size_mean']]
    size_df = size_df.rename(columns={'Size_mean': 'size'})
    size_df['uniqueID'] = size_df.uniqueID.str.replace(' ', '_')

    obj12_df = pd.merge(
        left=obj_df, right=size_df, on='uniqueID', how='outer',
    ) # 12 object properties
    assert obj12_df.shape[0] == obj_df.shape[0] == size_df.shape[0]
#     obj12_df.head()
    return obj12_df

# load object properties ratings for entire THINGS object concept database
obj12_df = load_object_property() # (1854,13)
print(obj12_df.head())

          uniqueID  manmade  precious  animacy   heavy  natural   moves  \
0         aardvark   1.2632    4.6053   6.8421  4.7895   6.7105  6.9211   
1           abacus   6.8718    4.1026   1.5128  3.3590   2.2051  4.2308   
2        accordion   6.7500    4.5000   1.5000  5.0556   2.0000  4.6944   
3            acorn   1.2000    4.2000   5.0000  1.4857   6.8286  2.7143   
4  air_conditioner   6.8158    4.1579   1.2895  6.1842   1.3684  2.1842   

    grasp    hold  be_moved  pleasant  deprecatedArousing        size  
0  3.1579  3.1842    3.3158    4.3947              4.1579  273.743243  
1  5.5128  5.4359    5.6667    4.7436              3.7692  241.705882  
2  4.8889  4.9167    5.0833    5.0556              3.8889  260.434783  
3  6.5714  6.7429    6.7429    5.1143              3.5429  148.326087  
4  3.3947  2.6316    2.6842    5.7368              2.7632  281.923913  


In [5]:
from tqdm import *
import os
import pandas as pd
import numpy as np
from os.path import join as pjoin

# read functional data
basedir = r'/n02dat01/users/qdzhao/THINGS'
betas_csv_dir = pjoin(basedir, r'THINGS_fMRI/betas_csv')
sub = '03'

data_file = pjoin(betas_csv_dir, f'sub-{sub}_ResponseData.h5')
responses = pd.read_hdf(data_file)
stim_f = pjoin(betas_csv_dir, f'sub-{sub}_StimulusMetadata.csv')
stimdata = pd.read_csv(stim_f)

# average betas within concept
filenames = stimdata.loc[:]['stimulus']
concepts = np.array([
    fn[0:fn.rfind('_')]
    for fn in filenames
])
unique_concepts = np.unique(concepts)

betas_concept = []
for c in tqdm(unique_concepts):
    inds = np.where(concepts == c)
    inds = inds[0].tolist()
    betas_concept.append(responses.loc[:][inds].mean(axis=1))
betas_concept = np.stack(betas_concept)  # semantic concepts * voxel (720 * whole brain)


<ipython-input-5-acc5afd3eada>:13: FutureWarning: In a future version, the Index constructor will not infer numeric dtypes when passed object-dtype sequences (matching Series behavior)
  responses = pd.read_hdf(data_file)
100%|██████████| 720/720 [00:08<00:00, 84.94it/s]


In [6]:
import scipy.io as scio

betas_csv_dir = pjoin(basedir, r'THINGS_fMRI/betas_csv')
betas_concept_ = scio.savemat(pjoin(betas_csv_dir, f'sub-{sub}_betas_concept.mat'),{'data':betas_concept})

# betas_csv_dir = pjoin(basedir, r'THINGS_fMRI/betas_csv')
# betas_concept_ = scio.loadmat(pjoin(betas_csv_dir, f'sub-{sub}_betas_concept.mat'))

In [20]:
# 提取720物体从1854中index
ind_1854to720 = []
for con in unique_concepts:
    ind_1854to720.append(obj12_df.loc[obj12_df.uniqueID==con].index.tolist()[0])
print(len(ind_1854to720)) # 720

outdir = pjoin(basedir, r'animacy_analysis\derivatives\obj_pro_analysis_I')
scio.savemat(pjoin(outdir, f'index_1854to720.mat'),{'index_1854to720': ind_1854to720})


720


###  data for linear regression

In [21]:
# voxelwise encoding model for obj pro pcs
from sklearn.linear_model import RidgeCV,Ridge,Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import scipy.io as scio

print(betas_concept.shape) # (720,v)

obj_pro_pcs_ = scio.loadmat(pjoin(basedir, r'animacy_analysis\derivatives\obj_pro_analysis_I', f'obj_pro_pcs.mat'))
obj_pro_pcs__ = obj_pro_pcs_['obj_pro_pcs']
obj_pro_pcs = obj_pro_pcs__[ind_1854to720,:]
print(obj_pro_pcs.shape) # (720,5)

# fit linear regression with l2 regularization
prop_train,prop_test,betas_train,betas_test = train_test_split(obj_pro_pcs,betas_concept,test_size=0.1,random_state=10)
scio.savemat(pjoin(outdir, f'betas_concept_v2.mat'), {'betas_train':betas_train,'betas_test':betas_test})
scio.savemat(pjoin(outdir, f'obj720_pro_pcs.mat'), {'prop_train':prop_train,'prop_test':prop_test})


(720, 211339)
(720, 5)
